In [1]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/

In [2]:
from google.colab import files
import json
import os
import torch
import json
import random
from datetime import datetime
from typing import Dict, List
from unsloth import FastLanguageModel
import unsloth
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
print("загрузка train.jsonl, val.jsonl, test.jsonl")
uploaded = files.upload()
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data
train_data = load_jsonl("train.jsonl")
val_data = load_jsonl("val.jsonl")
test_data = load_jsonl("test.jsonl")

загрузка train.jsonl, val.jsonl, test.jsonl


Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl
Saving val.jsonl to val.jsonl
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [4]:
SYSTEM_PROMPT = """You are a fitness AI trainer. Create safe workouts in JSON format.

CRITICAL RULES:
1. Return ONLY valid JSON, no other text, no markdown
2. Use ONLY exercise_id from catalog
3. Every exercise MUST include ALL required fields

STRENGTH needs: exercise_id, exercise_type="strength", sets(1-10), reps(1-50), weight_kg
CARDIO needs: exercise_id, exercise_type="cardio", duration_minutes(1-120), pace(walk/jog/run/sprint/recovery)
YOGA needs: exercise_id, exercise_type="yoga", hold_seconds(5-300), breath_count(1-20)

Example: {"workout_name": "Leg Day", "type": "strength", "duration_min": 30, "exercises": [{"exercise_id": "squat", "exercise_type": "strength", "sets": 3, "reps": 10, "weight_kg": null}]}"""

def format_example(example: Dict) -> str:
    user_msg = example["messages"][1]["content"]
    assistant_msg = example["messages"][2]["content"]

    # Проверяем JSON
    try:
        json.loads(assistant_msg)
    except:
        if assistant_msg.startswith('"') and assistant_msg.endswith('"'):
            assistant_msg = assistant_msg[1:-1]

    return f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
{assistant_msg}<|im_end|>"""

formatted_train = [format_example(ex) for ex in train_data]
formatted_val = [format_example(ex) for ex in val_data] if val_data else []

train_dataset = Dataset.from_dict({"text": formatted_train})
val_dataset = Dataset.from_dict({"text": formatted_val}) if val_data else None

In [5]:
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

training_args = TrainingArguments(
    output_dir="./workout_model",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=10,
    logging_first_step=True,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps" if val_dataset else "no",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="tensorboard",
    seed=42,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset if val_dataset else None,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=2048,
)

trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/3768 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/520 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,768 | Num Epochs = 2 | Total steps = 472
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss,Validation Loss
100,0.118701,0.109862
200,0.099347,0.093360
300,0.083282,0.087012


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Step,Training Loss,Validation Loss
100,0.118701,0.109862
200,0.099347,0.093360
300,0.083282,0.087012
400,0.082541,0.082535
472,0.082269,0.081872


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=472, training_loss=0.10251310347753056, metrics={'train_runtime': 12796.7102, 'train_samples_per_second': 0.589, 'train_steps_per_second': 0.037, 'total_flos': 1.6101305012124058e+17, 'train_loss': 0.10251310347753056, 'epoch': 2.0})

In [8]:
model.save_pretrained("workout_lora_model")
tokenizer.save_pretrained("workout_lora_model")
!zip -r workout_lora_model.zip workout_lora_model/
files.download("workout_lora_model.zip")

  adding: workout_lora_model/ (stored 0%)
  adding: workout_lora_model/adapter_config.json (deflated 57%)
  adding: workout_lora_model/adapter_model.safetensors (deflated 7%)
  adding: workout_lora_model/tokenizer.json (deflated 81%)
  adding: workout_lora_model/tokenizer_config.json (deflated 43%)
  adding: workout_lora_model/README.md (deflated 65%)
  adding: workout_lora_model/chat_template.jinja (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>